# 02 Baselines

Fixed-time, fixed-cycle, Max-Pressure and independent Q-learning on the default demand.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

RAW = ROOT / "results" / "raw"
FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
METHODS = ["fixed_time", "fixed_cycle", "max_pressure", "ql"]
LABELS = {"fixed_time": "Fixed-time", "fixed_cycle": "Fixed-cycle", "max_pressure": "Max-Pressure", "ql": "Independent QL"}
COLORS = {"fixed_time": "#0072B2", "fixed_cycle": "#56B4E9", "max_pressure": "#009E73", "ql": "#E69F00"}
METRICS = {
    "avg_waiting_time": "Avg waiting time (s)",
    "avg_queue": "Avg queue (veh)",
    "avg_trip_duration": "Avg trip duration (s)",
    "avg_time_loss": "Avg time loss (s)",
    "throughput": "Throughput (veh)",
    "avg_speed": "Avg speed (m/s)",
    "teleports": "Teleports (veh)",
}
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
frames = []
for scenario_dir in sorted(p for p in RAW.iterdir() if p.is_dir() and (p / "default").is_dir()):
    for method in METHODS:
        path = scenario_dir / "default" / f"{method}.csv"
        if path.is_file():
            frames.append(pd.read_csv(path))
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(df.groupby(["scenario", "method"]).size() if not df.empty else "No baseline CSVs found yet. Run scripts/run_baselines.py first.")

In [ ]:
def mean_ci(values, confidence=0.95):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    n = len(values)
    if n == 0:
        return np.nan, np.nan, 0
    mean = values.mean()
    if n < 2:
        return mean, np.nan, n
    half = stats.t.ppf((1 + confidence) / 2, n - 1) * values.std(ddof=1) / np.sqrt(n)
    return mean, half, n


def summarize(frame):
    rows = []
    for (scenario, method), group in frame.groupby(["scenario", "method"]):
        if group["train_seed"].nunique() > 1:
            units = group.groupby("train_seed")[list(METRICS)].mean()
        else:
            units = group[list(METRICS)]
        row = {"scenario": scenario, "method": method, "n": len(units)}
        for metric in METRICS:
            mean, half, _ = mean_ci(units[metric])
            row[f"{metric}_mean"] = mean
            row[f"{metric}_ci95"] = half
        rows.append(row)
    return pd.DataFrame(rows)


summary = summarize(df) if not df.empty else pd.DataFrame()
for scenario, block in summary.groupby("scenario"):
    table = pd.DataFrame({"method": block["method"].map(LABELS), "n": block["n"]})
    for metric in METRICS:
        table[metric] = [
            f"{m:.2f} ± {h:.2f}" if not np.isnan(h) else f"{m:.2f}"
            for m, h in zip(block[f"{metric}_mean"], block[f"{metric}_ci95"])
        ]
    print(f"\n{scenario}")
    display(table.set_index("method"))

In [ ]:
if not summary.empty:
    scenarios = sorted(summary["scenario"].unique())
    present = [m for m in METHODS if m in set(summary["method"])]
    width = 0.8 / max(1, len(present))
    x = np.arange(len(scenarios))
    fig, ax = plt.subplots(figsize=(7, 4))
    for i, method in enumerate(present):
        block = summary[summary["method"] == method].set_index("scenario").reindex(scenarios)
        ax.bar(
            x + i * width - 0.4 + width / 2,
            block["avg_waiting_time_mean"],
            width,
            yerr=block["avg_waiting_time_ci95"].fillna(0),
            capsize=3,
            label=LABELS[method],
            color=COLORS[method],
        )
    ax.set_xticks(x)
    ax.set_xticklabels(scenarios)
    ax.set_xlabel("Scenario")
    ax.set_ylabel("Avg waiting time per vehicle (s)")
    ax.set_title("Baselines on default demand (mean ± 95% CI)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / "02_baselines_waiting_time.png", dpi=300)
    plt.show()

In [ ]:
curves = sorted((RAW / "train_curves").glob("ql_*_seed*.csv"))
if curves:
    fig, ax = plt.subplots(figsize=(7, 4))
    for path in curves:
        curve = pd.read_csv(path)
        ax.plot(curve["episode"], curve["mean_waiting_time"], label=path.stem.removeprefix("ql_"))
    ax.set_xlabel("Training episode (3600 s each)")
    ax.set_ylabel("Mean system waiting time (s)")
    ax.set_title("Independent Q-learning training curve")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / "02_ql_training_curve.png", dpi=300)
    plt.show()
else:
    print("No Q-learning curves yet. Run python src/baselines/independent_ql.py first.")